<a href="https://colab.research.google.com/github/tshiams2025/-Mondayflowchart/blob/main/Wan2GP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Clone**

In [ ]:
%cd /content
!git clone https://github.com/deepbeepmeep/Wan2GP.git
%cd Wan2GP
!pip install -r requirements.txt

/content
Cloning into 'Wan2GP'...
remote: Enumerating objects: 17740, done.
remote: Counting objects: 100% (1349/1349), done.
remote: Compressing objects: 100% (682/682), done.
remote: Total 17740 (delta 825), reused 885 (delta 666), pack-reused 16391 (from 3)
Receiving objects: 100% (17740/17740), 90.76 MiB | 28.16 MiB/s, done.
Resolving deltas: 100% (11322/11322), done.
/content/Wan2GP
Looking in indexes: https://pypi.org/simple, https://aiinfra.pkgs.visualstudio.com/PublicPackages/_packaging/ort-cuda-13-nightly/pypi/simple/
Ignoring rembg: markers 'platform_system == "Darwin"' don't match your environment
Ignoring onnxruntime-gpu: markers 'python_version < "3.11" and platform_system != "Darwin"' don't match your environment
Ignoring onnxruntime: markers 'platform_system == "Darwin"' don't match your environment
Ignoring insightface: markers 'sys_platform == "win32" and python_version == "3.10"' don't match your environment
Ignoring insightface: markers 'sys_platform == "win32" and p

In [ ]:
!pip uninstall -y torch torchaudio torchvision
!pip cache purge
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128

Found existing installation: torch 2.11.0+cu128
Uninstalling torch-2.11.0+cu128:
  Successfully uninstalled torch-2.11.0+cu128
Found existing installation: torchaudio 2.11.0+cu128
Uninstalling torchaudio-2.11.0+cu128:
  Successfully uninstalled torchaudio-2.11.0+cu128
Found existing installation: torchvision 0.26.0+cu128
Uninstalling torchvision-0.26.0+cu128:
  Successfully uninstalled torchvision-0.26.0+cu128
Files removed: 18
Looking in indexes: https://download.pytorch.org/whl/cu128
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 820.3/820.3 MB 752.7 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 25.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 59.7 MB/s eta 0:00:00


# **Fixing Some Errors**

In [ ]:
import sys
import os

# Check what's in the Python import path
print("sys.path:\n", '\n'.join(sys.path))

# Confirm file structure
!ls -R /content/Wan2GP/preprocessing/matanyone

sys.path:
 /content
/env/python
/usr/lib/python313.zip
/usr/lib/python3.13
/usr/lib/python3.13/lib-dynload

/usr/local/lib/python3.13/dist-packages
/usr/lib/python3/dist-packages
/usr/local/lib/python3.13/dist-packages/IPython/extensions
/root/.ipython
/content/Wan2GP/preprocessing/matanyone:
app.py	     matanyone_wrapper.py	 tutorial_single_target.mp4
__init__.py  tools			 utils
matanyone    tutorial_multi_targets.mp4

/content/Wan2GP/preprocessing/matanyone/matanyone:
config	inference  model

/content/Wan2GP/preprocessing/matanyone/matanyone/config:
eval_matanyone_config.yaml  hydra  __init__.py	model

/content/Wan2GP/preprocessing/matanyone/matanyone/config/hydra:
job_logging

/content/Wan2GP/preprocessing/matanyone/matanyone/config/hydra/job_logging:
custom-no-rank.yaml  custom.yaml

/content/Wan2GP/preprocessing/matanyone/matanyone/config/model:
base.yaml

/content/Wan2GP/preprocessing/matanyone/matanyone/inference:
image_feature_store.py	kv_memory_store.py  object_manager.py
infe

In [ ]:
# Replace 'TkAgg' with 'Agg' programmatically
file_path = "/content/Wan2GP/preprocessing/matanyone/tools/interact_tools.py"

with open(file_path, "r") as file:
    content = file.read()

content = content.replace("matplotlib.use('TkAgg')", "matplotlib.use('Agg')")

with open(file_path, "w") as file:
    file.write(content)

print("✅ Backend fixed to 'Agg' for headless environment.")


✅ Backend fixed to 'Agg' for headless environment.


# **Main Cell**

In [ ]:
# ============================================================
# YuE2 / Wan2GP T4 BF16 -> FP16 KV CACHE FIX
# Run this ONE cell, then restart/relaunch Wan2GP.
# ============================================================

import os
import re
import shutil

ROOT = "/content/Wan2GP"
ATTN = os.path.join(
    ROOT,
    "shared/llm_engines/nanovllm/layers/attention.py"
)

print("Checking:", ATTN)

if not os.path.exists(ATTN):
    raise FileNotFoundError(f"Cannot find: {ATTN}")

# Backup original file
backup = ATTN + ".backup_before_fp16_fix"

if not os.path.exists(backup):
    shutil.copy2(ATTN, backup)
    print("Backup created:", backup)
else:
    print("Backup already exists:", backup)

with open(ATTN, "r", encoding="utf-8") as f:
    text = f.read()

original = text

# ------------------------------------------------------------
# 1. Make KV-cache allocations FP16 where BF16 is hard-coded.
# ------------------------------------------------------------

patterns = [
    (
        r'(\bk_cache\b[^;\n]*?dtype\s*=\s*)torch\.bfloat16',
        r'\1torch.float16'
    ),
    (
        r'(\bv_cache\b[^;\n]*?dtype\s*=\s*)torch\.bfloat16',
        r'\1torch.float16'
    ),
    (
        r'(\bk_cache\b[^,\n]*,\s*)dtype\s*=\s*torch\.bfloat16',
        r'\1dtype=torch.float16'
    ),
    (
        r'(\bv_cache\b[^,\n]*,\s*)dtype\s*=\s*torch\.bfloat16',
        r'\1dtype=torch.float16'
    ),
]

for pattern, replacement in patterns:
    text = re.sub(pattern, replacement, text)

# ------------------------------------------------------------
# 2. Add a defensive dtype conversion inside store_kvcache().
#
# This guarantees that if the model produces FP16 K/V tensors,
# they cannot be written into a BF16 cache.
# ------------------------------------------------------------

marker = "def store_kvcache("

if marker in text:
    start = text.find(marker)

    # Find the function body start
    body_start = text.find(":", start)

    if body_start != -1:
        body_start += 1

        # Find first non-empty/non-comment line
        lines = text[body_start:].splitlines(True)

        insert_pos = body_start

        for line in lines:
            stripped = line.strip()
            if stripped and not stripped.startswith("#"):
                break
            insert_pos += len(line)

        # Detect indentation from first real line
        first_line = next(
            (x for x in lines if x.strip() and not x.strip().startswith("#")),
            ""
        )

        indent = re.match(r"\s*", first_line).group(0) if first_line else "    "

        guard = (
            f"\n{indent}# YuE2 T4 FP16 KV-cache compatibility fix\n"
            f"{indent}if k_cache.dtype != k.dtype:\n"
            f"{indent}{indent}k_cache = k_cache.to(dtype=k.dtype)\n"
            f"{indent}if v_cache.dtype != v.dtype:\n"
            f"{indent}{indent}v_cache = v_cache.to(dtype=v.dtype)\n"
        )

        # Only insert if this guard isn't already present
        if "YuE2 T4 FP16 KV-cache compatibility fix" not in text:
            text = text[:insert_pos] + guard + text[insert_pos:]
            print("Added defensive KV-cache dtype guard.")
        else:
            print("KV-cache dtype guard already present.")

# ------------------------------------------------------------
# 3. Write patched file
# ------------------------------------------------------------

if text != original:
    with open(ATTN, "w", encoding="utf-8") as f:
        f.write(text)

    print("\n✅ PATCH APPLIED")
else:
    print("\n⚠️ No direct BF16 cache allocation was changed.")
    print("The defensive guard was checked/added if possible.")

# ------------------------------------------------------------
# 4. Show relevant lines for verification
# ------------------------------------------------------------

print("\n--- KV CACHE CODE ---")

with open(ATTN, "r", encoding="utf-8") as f:
    lines = f.readlines()

for i, line in enumerate(lines, 1):
    if (
        "store_kvcache" in line
        or "YuE2 T4 FP16" in line
        or "k_cache" in line and "dtype" in line
        or "v_cache" in line and "dtype" in line
    ):
        print(f"{i}: {line.rstrip()}")

# ------------------------------------------------------------
# 5. Verify CUDA / T4
# ------------------------------------------------------------

import torch

print("\n--- CUDA ---")
print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO CUDA")
print("BF16 supported:", torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False)

print("\n============================================================")
print("✅ YuE2 FP16 KV-cache patch complete.")
print("Now stop/relaunch the Wan2GP app and test YuE2 again.")
print("============================================================")

Checking: /content/Wan2GP/shared/llm_engines/nanovllm/layers/attention.py
Backup created: /content/Wan2GP/shared/llm_engines/nanovllm/layers/attention.py.backup_before_fp16_fix
Added defensive KV-cache dtype guard.

✅ PATCH APPLIED

--- KV CACHE CODE ---
67:     def store_kvcache_kernel(
456: def store_kvcache(
458:  # YuE2 T4 FP16 KV-cache compatibility fix
459:  if k_cache.dtype != k.dtype:
460:   k_cache = k_cache.to(dtype=k.dtype)
461:  if v_cache.dtype != v.dtype:
462:   v_cache = v_cache.to(dtype=v.dtype)
473:     quantized = k_cache.dtype == torch.int8
493:         store_kvcache_kernel[(N,)](key, key.stride(0), value, value.stride(0), k_cache, v_cache, slot_mapping, D)
581:             store_kvcache(k, v, k_cache, v_cache, context.slot_mapping, self.k_scale, self.v_scale, use_triton_kv_cache=self.use_triton_kv_cache)
582:         quantized_cache = k_cache.dtype == torch.int8
620:         attention_k_cache = _dequantize_kvcache(k_cache, self.k_scale, q.dtype) if quantized_cache a

In [ ]:
# ============================================================
# FIX BROKEN attention.py + SAFE YuE2 FP16 KV CACHE PATCH
# ============================================================

import os
import shutil
import re

ROOT = "/content/Wan2GP"
ATTN = f"{ROOT}/shared/llm_engines/nanovllm/layers/attention.py"
BACKUP = ATTN + ".backup_before_fp16_fix"

print("Target:", ATTN)

# ------------------------------------------------------------
# 1. Restore the clean original file from our backup
# ------------------------------------------------------------

if not os.path.exists(BACKUP):
    raise FileNotFoundError(
        "Backup not found. Please run the original Wan2GP checkout "
        "or show me the output of: git status"
    )

shutil.copy2(BACKUP, ATTN)
print("✅ Restored clean attention.py")

# ------------------------------------------------------------
# 2. Inspect store_kvcache()
# ------------------------------------------------------------

with open(ATTN, "r", encoding="utf-8") as f:
    text = f.read()

start = text.find("def store_kvcache(")

if start == -1:
    raise RuntimeError("store_kvcache() was not found.")

end = text.find("\ndef ", start + 5)
if end == -1:
    end = min(start + 5000, len(text))

print("\n--- store_kvcache() ---")
print(text[start:end])

# ------------------------------------------------------------
# 3. IMPORTANT:
# Instead of modifying cache allocation, make the source K/V
# tensors match the cache dtype immediately before assignment.
#
# Original failing line:
#
# flat_k_cache[slot_ids] = key[valid_mask]
#
# We change it to:
#
# key_to_store = key[valid_mask].to(flat_k_cache.dtype)
# flat_k_cache[slot_ids] = key_to_store
#
# Same for V.
# ------------------------------------------------------------

old_k = "flat_k_cache[slot_ids] = key[valid_mask]"
old_v = "flat_v_cache[slot_ids] = value[valid_mask]"

new_k = """flat_k_cache[slot_ids] = key[valid_mask].to(dtype=flat_k_cache.dtype)"""
new_v = """flat_v_cache[slot_ids] = value[valid_mask].to(dtype=flat_v_cache.dtype)"""

changed = False

if old_k in text:
    text = text.replace(old_k, new_k)
    print("✅ Patched K cache dtype conversion")
    changed = True
else:
    print("⚠️ K cache assignment not found exactly")

if old_v in text:
    text = text.replace(old_v, new_v)
    print("✅ Patched V cache dtype conversion")
    changed = True
else:
    print("⚠️ V cache assignment not found exactly")

# ------------------------------------------------------------
# 4. Write the safe patch
# ------------------------------------------------------------

with open(ATTN, "w", encoding="utf-8") as f:
    f.write(text)

# ------------------------------------------------------------
# 5. Python syntax check BEFORE launching Wan2GP
# ------------------------------------------------------------

print("\nChecking Python syntax...")

import py_compile

try:
    py_compile.compile(
        ATTN,
        doraise=True
    )
    print("✅ attention.py syntax is valid")
except Exception as e:
    print("❌ Syntax error:")
    print(e)
    raise

# ------------------------------------------------------------
# 6. Verify exact patched lines
# ------------------------------------------------------------

print("\n--- PATCHED CACHE ASSIGNMENTS ---")

with open(ATTN, "r", encoding="utf-8") as f:
    for i, line in enumerate(f, 1):
        if "flat_k_cache[slot_ids]" in line or "flat_v_cache[slot_ids]" in line:
            print(f"{i}: {line.rstrip()}")

# ------------------------------------------------------------
# 7. CUDA information
# ------------------------------------------------------------

try:
    import torch

    print("\n--- GPU ---")
    print("PyTorch:", torch.__version__)
    print("CUDA:", torch.version.cuda)

    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
        print("BF16 supported:", torch.cuda.is_bf16_supported())
        print("FP16 supported: True")

except Exception as e:
    print("Torch check skipped:", e)

print("\n============================================================")
print("✅ SAFE PATCH COMPLETE")
print("============================================================")
print()
print("The previous broken patch has been removed.")
print("The KV cache now converts K/V to the cache dtype")
print("before the indexed assignment.")
print()
print("Now RESTART Wan2GP and test YuE2.")

Target: /content/Wan2GP/shared/llm_engines/nanovllm/layers/attention.py
✅ Restored clean attention.py

--- store_kvcache() ---
def store_kvcache(
    key: torch.Tensor,
    value: torch.Tensor,
    k_cache: torch.Tensor,
    v_cache: torch.Tensor,
    slot_mapping: torch.Tensor,
    k_scale: torch.Tensor | None = None,
    v_scale: torch.Tensor | None = None,
    use_triton_kv_cache: bool | None = None,
):
    assert slot_mapping.numel() == key.shape[0]
    quantized = k_cache.dtype == torch.int8
    if quantized and (k_scale is None or v_scale is None):
        raise RuntimeError("INT8 KV cache requires key and value scales.")
    if use_triton_kv_cache is None:
        use_triton_kv_cache = _USE_TRITON_KV_CACHE
    if use_triton_kv_cache:
        N, num_heads, head_dim = key.shape
        if quantized:
            if head_dim % _Q8_KV_BLOCK_SIZE:
                raise RuntimeError(f"INT8 KV cache head dimension must be divisible by {_Q8_KV_BLOCK_SIZE}; got {head_dim}.")
            q

In [ ]:
# ============================================================
# YuE2 / Wan2GP T4 FIX
# FORCE NANOvLLM KV CACHE TO FP16
# ============================================================

import os
import re
import shutil
import py_compile

ROOT = "/content/Wan2GP"
ATTN = f"{ROOT}/shared/llm_engines/nanovllm/layers/attention.py"
BACKUP = ATTN + ".backup_before_fp16_fix"

print("============================================================")
print("YuE2 T4 FP16 KV-CACHE FIX")
print("============================================================")

# ------------------------------------------------------------
# Restore clean file from backup
# ------------------------------------------------------------

if os.path.exists(BACKUP):
    shutil.copy2(BACKUP, ATTN)
    print("✅ Restored clean attention.py from backup")
else:
    print("⚠️ Backup not found - using current attention.py")

with open(ATTN, "r", encoding="utf-8") as f:
    text = f.read()

# ------------------------------------------------------------
# Find all KV-cache allocations
# ------------------------------------------------------------

print("\nSearching for KV cache allocations...")

lines = text.splitlines()

for i, line in enumerate(lines, 1):
    low = line.lower()

    if (
        ("k_cache" in low or "v_cache" in low)
        and ("empty" in low or "zeros" in low or "dtype" in low)
    ):
        print(f"{i}: {line}")

# ------------------------------------------------------------
# Target the common NanoVLLM cache allocation patterns.
# We ONLY change cache dtype, not the rest of YuE2.
# ------------------------------------------------------------

replacements = [
    (
        r'(k_cache\s*=\s*torch\.(?:empty|zeros)\([^)]*?dtype\s*=\s*)torch\.bfloat16',
        r'\1torch.float16'
    ),
    (
        r'(v_cache\s*=\s*torch\.(?:empty|zeros)\([^)]*?dtype\s*=\s*)torch\.bfloat16',
        r'\1torch.float16'
    ),
    (
        r'(k_cache\s*=\s*torch\.(?:empty|zeros)\([^)]*)',
        r'\1'
    ),
]

changed = 0

for pattern, replacement in replacements[:2]:
    new_text, count = re.sub(
        pattern,
        replacement,
        text,
        flags=re.DOTALL
    )

    if count:
        text = new_text
        changed += count

# ------------------------------------------------------------
# More robust line-based replacement for cache declarations.
# ------------------------------------------------------------

out = []

for line in text.splitlines(True):

    stripped = line.lower()

    if (
        ("k_cache" in stripped or "v_cache" in stripped)
        and "dtype=torch.bfloat16" in stripped
    ):
        line = line.replace(
            "dtype=torch.bfloat16",
            "dtype=torch.float16"
        )
        changed += 1

    if (
        ("k_cache" in stripped or "v_cache" in stripped)
        and "dtype = torch.bfloat16" in stripped
    ):
        line = line.replace(
            "dtype = torch.bfloat16",
            "dtype = torch.float16"
        )
        changed += 1

    out.append(line)

text = "".join(out)

# ------------------------------------------------------------
# IMPORTANT FALLBACK:
# Find the cache creation section and force dtype from the
# attention/model dtype when the cache is created.
# ------------------------------------------------------------

# Replace explicit BF16 cache dtype anywhere in the cache
# creation expressions only.
#
# This is intentionally limited to lines mentioning:
# k_cache / v_cache / key_cache / value_cache.

out = []

for line in text.splitlines(True):

    low = line.lower()

    is_cache_line = any(
        x in low
        for x in [
            "k_cache",
            "v_cache",
            "key_cache",
            "value_cache"
        ]
    )

    if is_cache_line and "bfloat16" in low:
        line = re.sub(
            r'torch\.bfloat16',
            'torch.float16',
            line
        )
        line = re.sub(
            r'torch\.dtype\(torch\.bfloat16\)',
            'torch.float16',
            line
        )

    out.append(line)

text = "".join(out)

# ------------------------------------------------------------
# Write patch
# ------------------------------------------------------------

with open(ATTN, "w", encoding="utf-8") as f:
    f.write(text)

print(f"\nChanged cache-related BF16 declarations: {changed}")

# ------------------------------------------------------------
# Syntax check
# ------------------------------------------------------------

print("\nChecking Python syntax...")

py_compile.compile(
    ATTN,
    doraise=True
)

print("✅ attention.py syntax OK")

# ------------------------------------------------------------
# Show every remaining cache-related BF16 occurrence
# ------------------------------------------------------------

print("\n--- REMAINING BF16 CACHE REFERENCES ---")

found = False

with open(ATTN, "r", encoding="utf-8") as f:
    for i, line in enumerate(f, 1):

        low = line.lower()

        if (
            "bfloat16" in low
            and any(
                x in low
                for x in [
                    "k_cache",
                    "v_cache",
                    "key_cache",
                    "value_cache"
                ]
            )
        ):
            print(f"{i}: {line.rstrip()}")
            found = True

if not found:
    print("✅ No BF16 KV-cache allocation found")

# ------------------------------------------------------------
# Show cache-related lines
# ------------------------------------------------------------

print("\n--- CACHE LINES ---")

with open(ATTN, "r", encoding="utf-8") as f:
    for i, line in enumerate(f, 1):

        low = line.lower()

        if any(
            x in low
            for x in [
                "k_cache",
                "v_cache",
                "key_cache",
                "value_cache"
            ]
        ):
            if (
                "torch.empty" in low
                or "torch.zeros" in low
                or "dtype" in low
                or "cache" in low
            ):
                print(f"{i}: {line.rstrip()}")

# ------------------------------------------------------------
# GPU check
# ------------------------------------------------------------

try:
    import torch

    print("\n--- GPU ---")
    print("PyTorch:", torch.__version__)
    print("CUDA:", torch.version.cuda)

    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
        print(
            "BF16 supported:",
            torch.cuda.is_bf16_supported()
        )

except Exception as e:
    print("GPU check:", e)

print("\n============================================================")
print("✅ PATCH COMPLETE")
print("============================================================")
print("Restart Wan2GP completely before testing YuE2.")
print("The YuE2 AR query/key/value path should now use FP16.")

YuE2 T4 FP16 KV-CACHE FIX
✅ Restored clean attention.py from backup

Searching for KV cache allocations...
467:     quantized = k_cache.dtype == torch.int8
576:         quantized_cache = k_cache.dtype == torch.int8
614:         attention_k_cache = _dequantize_kvcache(k_cache, self.k_scale, q.dtype) if quantized_cache and reads_cache else k_cache
615:         attention_v_cache = _dequantize_kvcache(v_cache, self.v_scale, q.dtype) if quantized_cache and reads_cache else v_cache


ValueError: too many values to unpack (expected 2)

In [ ]:
# ============================================================
# YuE2 / Wan2GP - SAFE T4 FP16 KV CACHE FIX v2
# ============================================================

import os
import re
import shutil
import py_compile

ROOT = "/content/Wan2GP"
NANO = f"{ROOT}/shared/llm_engines/nanovllm"
ATTN = f"{NANO}/layers/attention.py"
BACKUP = ATTN + ".backup_before_fp16_fix"

print("=" * 60)
print("YuE2 / NanoVLLM T4 KV CACHE DIAGNOSTIC + FIX")
print("=" * 60)

# ------------------------------------------------------------
# 1. Restore attention.py to clean state
# ------------------------------------------------------------

if os.path.exists(BACKUP):
    shutil.copy2(BACKUP, ATTN)
    print("✅ attention.py restored from clean backup")
else:
    print("⚠️ No backup found; using current attention.py")

# ------------------------------------------------------------
# 2. Search NanoVLLM for KV cache creation
# ------------------------------------------------------------

print("\nSearching NanoVLLM for KV cache allocation...")

matches = []

for root, dirs, files in os.walk(NANO):
    for filename in files:
        if not filename.endswith(".py"):
            continue

        path = os.path.join(root, filename)

        try:
            with open(path, "r", encoding="utf-8") as f:
                lines = f.readlines()
        except Exception:
            continue

        for i, line in enumerate(lines, 1):

            low = line.lower()

            # Look for actual cache creation / dtype declarations
            if any(x in low for x in [
                "k_cache",
                "v_cache",
                "key_cache",
                "value_cache",
                "kv_cache",
                "cache_dtype",
                "dtype"
            ]):

                if any(x in low for x in [
                    "torch.empty",
                    "torch.zeros",
                    "torch.full",
                    "bfloat16",
                    "float16",
                    "cache_dtype",
                    "kv_cache"
                ]):

                    matches.append((path, i, line.rstrip()))

print(f"Found {len(matches)} potentially relevant lines.\n")

for path, line_no, line in matches:
    print(f"{path}:{line_no}")
    print("   ", line)

# ------------------------------------------------------------
# 3. Identify files that explicitly contain BF16 cache dtype
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("EXPLICIT BF16 CACHE REFERENCES")
print("=" * 60)

bf16_matches = []

for root, dirs, files in os.walk(NANO):
    for filename in files:

        if not filename.endswith(".py"):
            continue

        path = os.path.join(root, filename)

        try:
            with open(path, "r", encoding="utf-8") as f:
                lines = f.readlines()
        except Exception:
            continue

        for i, line in enumerate(lines, 1):

            low = line.lower()

            if "bfloat16" in low:

                # Show BF16 lines that are likely connected to cache
                nearby = " ".join(
                    x.lower()
                    for x in lines[max(0, i-4):min(len(lines), i+4)]
                )

                if any(x in nearby for x in [
                    "cache",
                    "k_cache",
                    "v_cache",
                    "key_cache",
                    "value_cache",
                    "kv"
                ]):

                    bf16_matches.append(
                        (path, i, line.rstrip())
                    )

for path, line_no, line in bf16_matches:
    print(f"{path}:{line_no}")
    print("   ", line)

if not bf16_matches:
    print("No explicit BF16 cache allocation found.")

# ------------------------------------------------------------
# 4. SAFELY patch only lines that are clearly cache-related
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("APPLYING SAFE PATCH")
print("=" * 60)

patched_files = []

for root, dirs, files in os.walk(NANO):

    for filename in files:

        if not filename.endswith(".py"):
            continue

        path = os.path.join(root, filename)

        # Never modify backup files
        if ".backup" in filename:
            continue

        try:
            with open(path, "r", encoding="utf-8") as f:
                original = f.read()
        except Exception:
            continue

        lines = original.splitlines(True)
        output = []

        changed = False

        for idx, line in enumerate(lines):

            low = line.lower()

            # Only modify a line when it contains BOTH:
            #   cache reference
            #   BF16 dtype
            #
            # This prevents touching unrelated YuE2 BF16 models.

            cache_related = any(x in low for x in [
                "k_cache",
                "v_cache",
                "key_cache",
                "value_cache",
                "kv_cache"
            ])

            has_bf16 = "torch.bfloat16" in low

            if cache_related and has_bf16:

                new_line = line.replace(
                    "torch.bfloat16",
                    "torch.float16"
                )

                if new_line != line:
                    line = new_line
                    changed = True

            output.append(line)

        if changed:

            # Make backup before modification
            backup = path + ".backup_yue2_fp16"

            if not os.path.exists(backup):
                shutil.copy2(path, backup)

            with open(path, "w", encoding="utf-8") as f:
                f.write("".join(output))

            patched_files.append(path)

            print("✅ Patched:", path)

# ------------------------------------------------------------
# 5. Syntax check all modified files
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("PYTHON SYNTAX CHECK")
print("=" * 60)

for path in patched_files:

    try:
        py_compile.compile(
            path,
            doraise=True
        )
        print("✅", path)

    except Exception as e:

        print("❌ Syntax error:", path)
        print(e)

        raise

# Always check attention.py
py_compile.compile(
    ATTN,
    doraise=True
)

print("✅ attention.py syntax OK")

# ------------------------------------------------------------
# 6. Show actual cache-related code around matches
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("CACHE CODE AFTER PATCH")
print("=" * 60)

for path, line_no, line in matches:

    if not os.path.exists(path):
        continue

    try:
        with open(path, "r", encoding="utf-8") as f:
            lines = f.readlines()
    except Exception:
        continue

    # Only display lines around actual cache references
    low = line.lower()

    if any(x in low for x in [
        "k_cache",
        "v_cache",
        "key_cache",
        "value_cache",
        "kv_cache"
    ]):

        start = max(0, line_no - 3)
        end = min(len(lines), line_no + 3)

        print(f"\n--- {path}:{line_no} ---")

        for n in range(start, end):
            print(f"{n+1}: {lines[n].rstrip()}")

# ------------------------------------------------------------
# 7. GPU information
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("GPU")
print("=" * 60)

import torch

print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)

if torch.cuda.is_available():

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

    print(
        "BF16 supported:",
        torch.cuda.is_bf16_supported()
    )

    print("FP16: supported")

print("\n" + "=" * 60)
print("DONE")
print("=" * 60)

if patched_files:
    print("✅ Cache-related BF16 declarations were patched.")
else:
    print("⚠️ No explicit BF16 KV-cache allocation was found.")
    print("The diagnostic output above shows where the cache is handled.")

print("\nDO NOT start Wan2GP yet if the cell reports an error.")
print("If syntax is OK, restart Wan2GP and test YuE2.")

YuE2 / NanoVLLM T4 KV CACHE DIAGNOSTIC + FIX
✅ attention.py restored from clean backup

Searching NanoVLLM for KV cache allocation...
Found 147 potentially relevant lines.

/content/Wan2GP/shared/llm_engines/nanovllm/config.py:19
        kv_cache_int8: bool = False
/content/Wan2GP/shared/llm_engines/nanovllm/config.py:20
        kv_cache_initial_tokens: int = 0
/content/Wan2GP/shared/llm_engines/nanovllm/config.py:21
        kv_cache_max_tokens: int = 0
/content/Wan2GP/shared/llm_engines/nanovllm/config.py:22
        kv_cache_prompt_tokens: int = 0
/content/Wan2GP/shared/llm_engines/nanovllm/vllm_support.py:182
        def __init__(self, model, model_path: str, tokenizer, enforce_eager: bool = False, graph_pool_handle=None, kv_cache_int8: bool = False):
/content/Wan2GP/shared/llm_engines/nanovllm/vllm_support.py:188
            self.kv_cache_int8 = bool(kv_cache_int8)
/content/Wan2GP/shared/llm_engines/nanovllm/vllm_support.py:189
            self._kv_cache_options = {}
/content/Wan2GP

In [ ]:
# ============================================================
# FINAL YuE2 T4 FIX
# Convert K/V to cache dtype BEFORE indexed assignment
# ============================================================

import os
import shutil
import py_compile

ATTN = "/content/Wan2GP/shared/llm_engines/nanovllm/layers/attention.py"

# ------------------------------------------------------------
# Backup current clean file
# ------------------------------------------------------------

backup = ATTN + ".backup_final_yue2"

if not os.path.exists(backup):
    shutil.copy2(ATTN, backup)

# ------------------------------------------------------------
# Read file
# ------------------------------------------------------------

with open(ATTN, "r", encoding="utf-8") as f:
    text = f.read()

# ------------------------------------------------------------
# Patch EXACT failing assignments
# ------------------------------------------------------------

old_k = "flat_k_cache[slot_ids] = key[valid_mask]"
old_v1 = "flat_v_cache[slot_ids] = value[valid_mask]"
old_v2 = "flat_v_cache[slot_ids] = val[valid_mask]"

new_k = "flat_k_cache[slot_ids] = key[valid_mask].to(dtype=flat_k_cache.dtype)"
new_v1 = "flat_v_cache[slot_ids] = value[valid_mask].to(dtype=flat_v_cache.dtype)"
new_v2 = "flat_v_cache[slot_ids] = val[valid_mask].to(dtype=flat_v_cache.dtype)"

changed = 0

if old_k in text:
    text = text.replace(old_k, new_k)
    print("✅ Patched K cache assignment")
    changed += 1
else:
    print("⚠️ K cache assignment not found")

if old_v1 in text:
    text = text.replace(old_v1, new_v1)
    print("✅ Patched V cache assignment")
    changed += 1

if old_v2 in text:
    text = text.replace(old_v2, new_v2)
    print("✅ Patched V cache assignment (val)")
    changed += 1

# ------------------------------------------------------------
# Write
# ------------------------------------------------------------

with open(ATTN, "w", encoding="utf-8") as f:
    f.write(text)

# ------------------------------------------------------------
# Syntax check
# ------------------------------------------------------------

print("\nChecking syntax...")

py_compile.compile(
    ATTN,
    doraise=True
)

print("✅ attention.py syntax is valid")

# ------------------------------------------------------------
# Show exact patched code
# ------------------------------------------------------------

print("\n--- FINAL PATCH ---")

with open(ATTN, "r", encoding="utf-8") as f:
    lines = f.readlines()

for i, line in enumerate(lines, 1):
    if (
        "flat_k_cache[slot_ids]" in line
        or "flat_v_cache[slot_ids]" in line
    ):
        print(f"{i}: {line.rstrip()}")

print("\n============================================================")
print(f"✅ DONE — {changed} cache assignment(s) patched")
print("============================================================")
print()
print("Restart the Wan2GP process completely.")
print("Then test YuE2 again.")

✅ Patched K cache assignment
✅ Patched V cache assignment

Checking syntax...
✅ attention.py syntax is valid

--- FINAL PATCH ---
519:     flat_k_cache[slot_ids] = key[valid_mask].to(dtype=flat_k_cache.dtype).to(dtype=flat_k_cache.dtype)
520:     flat_v_cache[slot_ids] = value[valid_mask].to(dtype=flat_v_cache.dtype).to(dtype=flat_v_cache.dtype)

✅ DONE — 2 cache assignment(s) patched

Restart the Wan2GP process completely.
Then test YuE2 again.


In [ ]:
# ============================================================
# YuE2 T4 FIX #2
# Cast KV cache to Q dtype before SDPA attention
# ============================================================

import os
import shutil
import py_compile

ATTN = "/content/Wan2GP/shared/llm_engines/nanovllm/layers/attention.py"

print("=" * 65)
print("YuE2 T4 — KV CACHE READ DTYPE FIX")
print("=" * 65)

# ------------------------------------------------------------
# Backup current working file
# ------------------------------------------------------------

backup = ATTN + ".backup_kv_read_fix"

if not os.path.exists(backup):
    shutil.copy2(ATTN, backup)
    print("✅ Backup created")
else:
    print("✅ Backup already exists")

# ------------------------------------------------------------
# Read file
# ------------------------------------------------------------

with open(ATTN, "r", encoding="utf-8") as f:
    text = f.read()

# ------------------------------------------------------------
# Locate current cache-read lines
# ------------------------------------------------------------

old_k = """attention_k_cache = _dequantize_kvcache(k_cache, self.k_scale, q.dtype) if quantized_cache and reads_cache else k_cache"""

old_v = """attention_v_cache = _dequantize_kvcache(v_cache, self.v_scale, q.dtype) if quantized_cache and reads_cache else v_cache"""

new_k = """attention_k_cache = (_dequantize_kvcache(k_cache, self.k_scale, q.dtype) if quantized_cache and reads_cache else k_cache).to(dtype=q.dtype)"""

new_v = """attention_v_cache = (_dequantize_kvcache(v_cache, self.v_scale, q.dtype) if quantized_cache and reads_cache else v_cache).to(dtype=q.dtype)"""

changed = 0

# ------------------------------------------------------------
# Patch K cache
# ------------------------------------------------------------

if old_k in text:
    text = text.replace(old_k, new_k)
    print("✅ Patched attention K cache → q.dtype")
    changed += 1
elif new_k in text:
    print("ℹ️ K cache patch already applied")
else:
    print("⚠️ K cache line not found")

# ------------------------------------------------------------
# Patch V cache
# ------------------------------------------------------------

if old_v in text:
    text = text.replace(old_v, new_v)
    print("✅ Patched attention V cache → q.dtype")
    changed += 1
elif new_v in text:
    print("ℹ️ V cache patch already applied")
else:
    print("⚠️ V cache line not found")

# ------------------------------------------------------------
# Write patched file
# ------------------------------------------------------------

with open(ATTN, "w", encoding="utf-8") as f:
    f.write(text)

# ------------------------------------------------------------
# Syntax check
# ------------------------------------------------------------

print("\nChecking Python syntax...")

py_compile.compile(
    ATTN,
    doraise=True
)

print("✅ attention.py syntax is valid")

# ------------------------------------------------------------
# Show patched lines
# ------------------------------------------------------------

print("\n--- CACHE READ CODE ---")

with open(ATTN, "r", encoding="utf-8") as f:
    lines = f.readlines()

for i, line in enumerate(lines, 1):
    if "attention_k_cache =" in line or "attention_v_cache =" in line:
        print(f"{i}: {line.rstrip()}")

print("\n" + "=" * 65)
print(f"✅ DONE — {changed} line(s) changed")
print("=" * 65)

print("""
NEXT:
1. Completely stop the current Wan2GP process.
2. Restart Wan2GP.
3. Generate YuE2 again.

The expected change is:

Q  = FP16
K  = FP16
V  = FP16

instead of:

Q  = FP16
K  = BF16
V  = BF16
""")

YuE2 T4 — KV CACHE READ DTYPE FIX
✅ Backup created
✅ Patched attention K cache → q.dtype
✅ Patched attention V cache → q.dtype

Checking Python syntax...
✅ attention.py syntax is valid

--- CACHE READ CODE ---
614:         attention_k_cache = (_dequantize_kvcache(k_cache, self.k_scale, q.dtype) if quantized_cache and reads_cache else k_cache).to(dtype=q.dtype)
615:         attention_v_cache = (_dequantize_kvcache(v_cache, self.v_scale, q.dtype) if quantized_cache and reads_cache else v_cache).to(dtype=q.dtype)

✅ DONE — 2 line(s) changed

NEXT:
1. Completely stop the current Wan2GP process.
2. Restart Wan2GP.
3. Generate YuE2 again.

The expected change is:

Q  = FP16
K  = FP16
V  = FP16

instead of:

Q  = FP16
K  = BF16
V  = BF16



In [ ]:
# ============================================================
# YuE2 T4 FIX #3
# Fix ALL KV-cache index_copy_ dtype mismatches
# ============================================================

import os
import shutil
import py_compile

ATTN = "/content/Wan2GP/shared/llm_engines/nanovllm/layers/attention.py"

print("=" * 65)
print("YuE2 T4 — FINAL KV CACHE WRITE FIX")
print("=" * 65)

# ------------------------------------------------------------
# Backup
# ------------------------------------------------------------

backup = ATTN + ".backup_index_copy_fix"

if not os.path.exists(backup):
    shutil.copy2(ATTN, backup)
    print("✅ Backup created")
else:
    print("✅ Backup already exists")

# ------------------------------------------------------------
# Read
# ------------------------------------------------------------

with open(ATTN, "r", encoding="utf-8") as f:
    text = f.read()

changed = 0

# ------------------------------------------------------------
# FIX K CACHE index_copy_
# ------------------------------------------------------------

old = "flat_k_cache.index_copy_(0, slot_ids, key)"
new = "flat_k_cache.index_copy_(0, slot_ids, key.to(dtype=flat_k_cache.dtype))"

if old in text:
    text = text.replace(old, new)
    print("✅ Patched K index_copy_")
    changed += 1
elif new in text:
    print("ℹ️ K index_copy_ already patched")
else:
    print("⚠️ K index_copy_ not found")

# ------------------------------------------------------------
# FIX V CACHE index_copy_
# ------------------------------------------------------------

old = "flat_v_cache.index_copy_(0, slot_ids, value)"
new = "flat_v_cache.index_copy_(0, slot_ids, value.to(dtype=flat_v_cache.dtype))"

if old in text:
    text = text.replace(old, new)
    print("✅ Patched V index_copy_")
    changed += 1
elif new in text:
    print("ℹ️ V index_copy_ already patched")
else:
    print("⚠️ V index_copy_ not found")

# Some versions use "val" instead of "value"
old = "flat_v_cache.index_copy_(0, slot_ids, val)"
new = "flat_v_cache.index_copy_(0, slot_ids, val.to(dtype=flat_v_cache.dtype))"

if old in text:
    text = text.replace(old, new)
    print("✅ Patched V index_copy_ (val)")
    changed += 1
elif new in text:
    print("ℹ️ V index_copy_ (val) already patched")

# ------------------------------------------------------------
# Write
# ------------------------------------------------------------

with open(ATTN, "w", encoding="utf-8") as f:
    f.write(text)

# ------------------------------------------------------------
# Syntax check
# ------------------------------------------------------------

print("\nChecking syntax...")

py_compile.compile(
    ATTN,
    doraise=True
)

print("✅ attention.py syntax is valid")

# ------------------------------------------------------------
# Show every KV cache write
# ------------------------------------------------------------

print("\n--- ALL KV CACHE WRITES ---")

with open(ATTN, "r", encoding="utf-8") as f:
    lines = f.readlines()

for i, line in enumerate(lines, 1):
    if (
        "flat_k_cache[" in line
        or "flat_v_cache[" in line
        or "flat_k_cache.index_copy_" in line
        or "flat_v_cache.index_copy_" in line
    ):
        print(f"{i}: {line.rstrip()}")

print("\n" + "=" * 65)
print(f"✅ DONE — {changed} new cache write fix(es)")
print("=" * 65)

print("""
IMPORTANT:
The previous fixes remain in place.

We now handle both cache-write mechanisms:

1. indexed assignment
   key/value → cache dtype

2. index_copy_
   key/value → cache dtype

And the cache-read path already converts:
   K/V → Q dtype

Restart Wan2GP completely and test YuE2 again.
""")

YuE2 T4 — FINAL KV CACHE WRITE FIX
✅ Backup created
✅ Patched K index_copy_
✅ Patched V index_copy_

Checking syntax...
✅ attention.py syntax is valid

--- ALL KV CACHE WRITES ---
506:         flat_k_cache.index_copy_(0, slot_ids, key.to(dtype=flat_k_cache.dtype))
507:         flat_v_cache.index_copy_(0, slot_ids, value.to(dtype=flat_v_cache.dtype))
519:     flat_k_cache[slot_ids] = key[valid_mask].to(dtype=flat_k_cache.dtype).to(dtype=flat_k_cache.dtype)
520:     flat_v_cache[slot_ids] = value[valid_mask].to(dtype=flat_v_cache.dtype).to(dtype=flat_v_cache.dtype)

✅ DONE — 2 new cache write fix(es)

IMPORTANT:
The previous fixes remain in place.

We now handle both cache-write mechanisms:

1. indexed assignment
   key/value → cache dtype

2. index_copy_
   key/value → cache dtype

And the cache-read path already converts:
   K/V → Q dtype

Restart Wan2GP completely and test YuE2 again.



In [ ]:
# ============================================================
# YuE2 T4 FIX #4
# Qwen3 LM HEAD dtype mismatch: BF16 hidden -> FP16 weights
# ============================================================

import os
import shutil
import py_compile

HEAD = "/content/Wan2GP/shared/llm_engines/nanovllm/layers/embed_head.py"

print("=" * 65)
print("YuE2 T4 — LM HEAD DTYPE FIX")
print("=" * 65)

# ------------------------------------------------------------
# Backup
# ------------------------------------------------------------

backup = HEAD + ".backup_yue2_dtype"

if not os.path.exists(backup):
    shutil.copy2(HEAD, backup)
    print("✅ Backup created")
else:
    print("✅ Backup already exists")

# ------------------------------------------------------------
# Read file
# ------------------------------------------------------------

with open(HEAD, "r", encoding="utf-8") as f:
    text = f.read()

# ------------------------------------------------------------
# Find exact F.linear call
# ------------------------------------------------------------

old = "logits = F.linear(x, self.weight)"

new = "logits = F.linear(x.to(dtype=self.weight.dtype), self.weight)"

if old in text:
    text = text.replace(old, new)
    print("✅ Patched LM head input dtype")
elif new in text:
    print("ℹ️ LM head patch already applied")
else:
    print("⚠️ Expected F.linear line was not found")

# ------------------------------------------------------------
# Write
# ------------------------------------------------------------

with open(HEAD, "w", encoding="utf-8") as f:
    f.write(text)

# ------------------------------------------------------------
# Syntax check
# ------------------------------------------------------------

print("\nChecking syntax...")

py_compile.compile(
    HEAD,
    doraise=True
)

print("✅ embed_head.py syntax is valid")

# ------------------------------------------------------------
# Show patched line
# ------------------------------------------------------------

print("\n--- PATCHED LM HEAD ---")

with open(HEAD, "r", encoding="utf-8") as f:
    for i, line in enumerate(f, 1):
        if "F.linear" in line:
            print(f"{i}: {line.rstrip()}")

print("\n" + "=" * 65)
print("✅ DONE")
print("=" * 65)

print("""
The LM head will now automatically match:

hidden states → LM-head weight dtype

For your current error:

hidden = BF16
weight = FP16

the hidden states will be converted to FP16 only at F.linear().

Restart Wan2GP completely and test YuE2 again.
""")

YuE2 T4 — LM HEAD DTYPE FIX
✅ Backup created
✅ Patched LM head input dtype

Checking syntax...
✅ embed_head.py syntax is valid

--- PATCHED LM HEAD ---
66:         logits = F.linear(x.to(dtype=self.weight.dtype), self.weight)

✅ DONE

The LM head will now automatically match:

hidden states → LM-head weight dtype

For your current error:

hidden = BF16
weight = FP16

the hidden states will be converted to FP16 only at F.linear().

Restart Wan2GP completely and test YuE2 again.



In [ ]:
ATTN = "/content/Wan2GP/shared/llm_engines/nanovllm/layers/attention.py"

with open(ATTN, "r", encoding="utf-8") as f:
    lines = f.readlines()

for i in range(285, 325):
    if i <= len(lines):
        print(f"{i:4}: {lines[i-1].rstrip()}")

 285:     return output
 286: 
 287: 
 288: def _repeat_kv(hidden_states: torch.Tensor, num_repeats: int) -> torch.Tensor:
 289:     if num_repeats == 1:
 290:         return hidden_states
 291:     return hidden_states.repeat_interleave(num_repeats, dim=1)
 292: 
 293: 
 294: def _sdpa_attention(
 295:     query_states: torch.Tensor,
 296:     key_states: torch.Tensor,
 297:     value_states: torch.Tensor,
 298:     scaling: float,
 299:     num_key_value_groups: int,
 300:     attention_bias: torch.Tensor | None = None,
 301:     is_causal: bool = True,
 302: ) -> torch.Tensor:
 303:     # Avoid enable_gqa here: on the local Torch 2.10 + CUDA 13 build it can pick
 304:     # a much higher-workspace backend than the explicit-repeat SDPA path.
 305:     key_states = _repeat_kv(key_states, num_key_value_groups)
 306:     value_states = _repeat_kv(value_states, num_key_value_groups)
 307:     if attention_bias is not None and attention_bias.dtype != torch.bool and attention_bias.dtype !=

In [ ]:
!python wgp.py --listen --share

[GGUF][llama.cpp CUDA] kernels unavailable, using fallback
Switching to FP16 models when possible as GPU architecture doesn't support optimed BF16 Kernels
Loaded plugin: Motion Designer (from motion_designer)
WanGP web authentication disabled.
Deepy Web app: http://0.0.0.0:7860/deepy/
* Running on local URL:  http://0.0.0.0:7860
* Running on public URL: https://33c7921576601a23e8.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
pose/dw-ll_ucoco_384.onnx: 100% 134M/134M [00:01<00:00, 133MB/s] 
pose/yolox_l.onnx: 100% 217M/217M [00:01<00:00, 154MB/s]
scribble/netG_A_latest.pth: 100% 17.2M/17.2M [00:00<00:00, 42.7MB/s]
flow/raft-things.pth: 100% 21.1M/21.1M [00:00<00:00, 52.5MB/s]
depth/depth_anything_v2_vitl.pth: 100% 1.34G/1.34G [00:08<00:00, 164MB/s]
config.json: 100% 1.60k/1.60k [00:00<00:00, 5.62MB/s]
feature_extra

In [4]:
import gradio as gr
print(gr.__version__)

6.31.0


In [7]:
def reverse_text(text):
    return text[::-1]

In [9]:
demo = gr.Interface(fn=reverse_text, inputs="text", outputs="text")
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c922d5e53b383a4e90.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
